In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1eR9ZZrRdJSrVF8i3RC63WseDMuAOCfqm", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Notebook 2: Escalation & Human-in-the-Loop

**Claude Certified Architect Prep** | Pod 5: Context Management & Reliability

---

In Notebook 1, we learned how to manage context windows and prevent information loss. But even with perfect context management, there are situations where **no amount of clever prompting will replace human judgment.**

A customer threatening legal action. A policy gap with no precedent. A $500,000 claim that could bankrupt a small business if handled incorrectly. These are not edge cases -- they are Tuesday afternoon for any production AI system.

The question is not whether your agent will encounter situations it cannot handle. The question is: **will it recognize them, and will it hand off gracefully?**

In this notebook, you will build a complete escalation framework: concrete thresholds for when to escalate, structured handoff summaries that let humans pick up where the agent left off, and a stratified sampling system for ongoing quality review.

**What you will build:**
- An escalation classifier with concrete, testable thresholds
- A structured handoff protocol with context preservation
- A stratified sampling system for human quality review
- A comparison showing why stratified sampling catches more errors than random sampling

**Prerequisites:** Notebook 1 (Context Window Management)

**Estimated time:** 50 minutes

[Ask the AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/context-management-reliability/practice/2/assistant)

In [ ]:
# ============================================================
# Setup: Install dependencies
# ============================================================
# Run this cell first. All packages are available on Colab by default.

!pip install -q matplotlib numpy

import json
import random
import textwrap
from typing import Any
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from enum import Enum
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

random.seed(42)

# Utility: pretty-print JSON
def pp(obj: Any) -> None:
    """Pretty-print a Python object as formatted JSON."""
    print(json.dumps(obj, indent=2, default=str))

print("Setup complete.")
print("No API keys required -- this notebook uses mock data throughout.")

In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 1: Why Does This Matter?

Imagine your AI agent handles 5,000 customer support requests per day. It resolves 4,800 of them perfectly. But among the remaining 200:

- 15 involve customers who explicitly asked for a human
- 8 involve claims above $100,000
- 3 involve threats of legal action
- 12 involve policy questions the agent has never seen before

If the agent treats all 200 the same way -- either stubbornly trying to resolve them or dumping them all to the human queue with no context -- the result is either **wrong decisions** or **overwhelmed human reviewers.**

The exam tests whether you can build systems that:
1. Distinguish situations requiring human judgment from those the agent can handle
2. Apply concrete, measurable thresholds (not vague heuristics)
3. Provide structured handoffs so humans do not start from scratch
4. Use stratified sampling to catch errors where they are most likely to hide

Let's build each of these, step by step.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 2: Building Intuition -- When Should an Agent Escalate?

Before writing any code, let's develop our intuition. Below are 10 real customer messages. For each one, decide: should the agent handle it, or escalate to a human?

In [ ]:
# ============================================================
# 2A: The judgment exercise
# ============================================================

test_messages = [
    {
        "id": 1,
        "message": "Can you check the status of my order #4821?",
        "claim_value": 89.99,
        "context": "Standard order tracking request."
    },
    {
        "id": 2,
        "message": "I want to speak to a manager right now.",
        "claim_value": 150.00,
        "context": "Customer is upset about a late delivery."
    },
    {
        "id": 3,
        "message": "My lawyer will be contacting you about the denied claim.",
        "claim_value": 45000.00,
        "context": "Customer's home insurance claim was denied."
    },
    {
        "id": 4,
        "message": "I need to update my shipping address.",
        "claim_value": 0,
        "context": "Routine address change."
    },
    {
        "id": 5,
        "message": "This is the third time I've called about the same issue!",
        "claim_value": 320.00,
        "context": "Customer has had 2 prior unresolved interactions."
    },
    {
        "id": 6,
        "message": "I need to file a claim for $250,000 in flood damage.",
        "claim_value": 250000.00,
        "context": "Major property damage claim."
    },
    {
        "id": 7,
        "message": "Can you explain what my deductible covers?",
        "claim_value": 0,
        "context": "General policy question, answer is in the FAQ."
    },
    {
        "id": 8,
        "message": "I'm going to post about this on social media if you don't fix it.",
        "claim_value": 500.00,
        "context": "Customer threatening public complaint."
    },
    {
        "id": 9,
        "message": "Does my policy cover groundwater seepage? I can't find it anywhere.",
        "claim_value": 28500.00,
        "context": "Policy document has no mention of groundwater seepage."
    },
    {
        "id": 10,
        "message": "Please cancel my subscription.",
        "claim_value": 0,
        "context": "Routine cancellation request."
    },
]

print("EXERCISE: For each message, decide -- HANDLE or ESCALATE?")
print("=" * 65)
for msg in test_messages:
    print(f"\n  Message #{msg['id']}: \"{msg['message']}\"")
    print(f"  Claim value: ${msg['claim_value']:,.2f}")
    print(f"  Context: {msg['context']}")
    print(f"  Your decision: ___________")

print("\n\nThink about your answers before running the next cell.")
print("What criteria did you use? Were they consistent and repeatable?")

In [ ]:
# ============================================================
# 2B: The problem with gut-feel decisions
# ============================================================

# Here's what happens when three different team members make escalation
# decisions based on "gut feel" rather than explicit criteria:

team_decisions = {
    "Alice (cautious)": {
        1: "HANDLE", 2: "ESCALATE", 3: "ESCALATE", 4: "HANDLE",
        5: "ESCALATE", 6: "ESCALATE", 7: "HANDLE", 8: "ESCALATE",
        9: "ESCALATE", 10: "HANDLE"
    },
    "Bob (aggressive)": {
        1: "HANDLE", 2: "HANDLE", 3: "ESCALATE", 4: "HANDLE",
        5: "HANDLE", 6: "ESCALATE", 7: "HANDLE", 8: "HANDLE",
        9: "HANDLE", 10: "HANDLE"
    },
    "Carol (moderate)": {
        1: "HANDLE", 2: "ESCALATE", 3: "ESCALATE", 4: "HANDLE",
        5: "HANDLE", 6: "ESCALATE", 7: "HANDLE", 8: "HANDLE",
        9: "ESCALATE", 10: "HANDLE"
    },
}

print("Three team members, same 10 messages, different decisions:")
print("=" * 65)
print(f"\n  {'Message':<10} {'Alice':<12} {'Bob':<12} {'Carol':<12} {'Agreement?'}")
print(f"  {'-'*10} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")

disagreements = 0
for msg_id in range(1, 11):
    a = team_decisions["Alice (cautious)"][msg_id]
    b = team_decisions["Bob (aggressive)"][msg_id]
    c = team_decisions["Carol (moderate)"][msg_id]
    agree = "YES" if a == b == c else "NO"
    if agree == "NO":
        disagreements += 1
    print(f"  #{msg_id:<9} {a:<12} {b:<12} {c:<12} {agree}")

print(f"\n  Disagreements: {disagreements}/10 ({disagreements*10}%)")
print(f"\n  This is the problem with vague escalation rules.")
print(f"  'Escalate when things get complex' means different things")
print(f"  to different people -- and different things to an AI agent")
print(f"  at 2 AM vs 2 PM.")

### The Core Insight

Vague rules like "escalate when uncertain" are **not testable, not auditable, and not consistent.** Production systems need **concrete thresholds** that produce the same decision regardless of who (or what) applies them.

Let's build those thresholds.

---

In [ ]:
#@title 🎧 Listen: Escalation Classifier
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_escalation_classifier.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 3: Core Implementation -- Escalation Criteria with Concrete Thresholds

We will build an `EscalationClassifier` that evaluates each incoming request against five categories of escalation triggers. Each category has a concrete, measurable threshold.

In [ ]:
# ============================================================
# 3A: Define the escalation categories and thresholds
# ============================================================

class EscalationPriority(Enum):
    """Priority levels for escalation -- higher number = more urgent."""
    NONE = 0          # Agent handles it
    LOW = 1           # Flag for later review
    MEDIUM = 2        # Escalate within 1 hour
    HIGH = 3          # Escalate within 15 minutes
    CRITICAL = 4      # Escalate immediately

@dataclass
class EscalationResult:
    """Result of evaluating a request against escalation criteria."""
    should_escalate: bool
    priority: EscalationPriority
    reasons: list[str]
    triggered_rules: list[str]
    confidence: float  # 0-1, how confident the classifier is

    def to_dict(self) -> dict:
        return {
            "should_escalate": self.should_escalate,
            "priority": self.priority.name,
            "priority_value": self.priority.value,
            "reasons": self.reasons,
            "triggered_rules": self.triggered_rules,
            "confidence": round(self.confidence, 3),
        }


class EscalationClassifier:
    """
    Evaluates incoming requests against concrete escalation thresholds.

    Five categories of escalation triggers:
    1. Safety concerns (threats, legal action, urgent situations)
    2. Explicit customer requests (asks for human/manager/supervisor)
    3. High-value decisions (claim/transaction above dollar threshold)
    4. Policy gaps (no authoritative source covers the situation)
    5. Repeated failures (N consecutive unsuccessful attempts)
    """

    # --- Configurable thresholds ---
    SAFETY_KEYWORDS = [
        "lawyer", "attorney", "legal action", "sue", "lawsuit",
        "threat", "harm", "danger", "emergency", "police",
        "regulator", "complaint to", "report you"
    ]

    HUMAN_REQUEST_KEYWORDS = [
        "manager", "supervisor", "human", "real person",
        "speak to someone", "talk to a person", "escalate",
        "representative"
    ]

    HIGH_VALUE_THRESHOLD = 100_000  # dollars

    CONFIDENCE_FLOOR = 0.70  # below this = policy gap

    MAX_FAILED_ATTEMPTS = 3

    def classify(self, request: dict) -> EscalationResult:
        """
        Evaluate a single request against all escalation criteria.

        Args:
            request: dict with keys:
                - message (str): the customer's message
                - claim_value (float): dollar value involved
                - prior_attempts (int): number of prior failed attempts
                - policy_match_confidence (float): 0-1, how well
                  existing policy covers this situation
                - context (str): additional context

        Returns:
            EscalationResult with priority and triggered rules
        """
        reasons = []
        triggered = []
        max_priority = EscalationPriority.NONE
        msg_lower = request.get("message", "").lower()

        # --- Rule 1: Safety concerns ---
        safety_matches = [
            kw for kw in self.SAFETY_KEYWORDS if kw in msg_lower
        ]
        if safety_matches:
            reasons.append(
                f"Safety concern detected: keywords [{', '.join(safety_matches)}]"
            )
            triggered.append("SAFETY_CONCERN")
            max_priority = max(max_priority, EscalationPriority.CRITICAL,
                               key=lambda p: p.value)

        # --- Rule 2: Customer explicitly requests human ---
        human_matches = [
            kw for kw in self.HUMAN_REQUEST_KEYWORDS if kw in msg_lower
        ]
        if human_matches:
            reasons.append(
                f"Customer requested human: keywords [{', '.join(human_matches)}]"
            )
            triggered.append("CUSTOMER_REQUESTS_HUMAN")
            max_priority = max(max_priority, EscalationPriority.HIGH,
                               key=lambda p: p.value)

        # --- Rule 3: High-value decision ---
        claim_value = request.get("claim_value", 0)
        if claim_value > self.HIGH_VALUE_THRESHOLD:
            reasons.append(
                f"High-value decision: ${claim_value:,.2f} exceeds "
                f"${self.HIGH_VALUE_THRESHOLD:,.2f} threshold"
            )
            triggered.append("HIGH_VALUE_DECISION")
            max_priority = max(max_priority, EscalationPriority.HIGH,
                               key=lambda p: p.value)

        # --- Rule 4: Policy gap ---
        confidence = request.get("policy_match_confidence", 1.0)
        if confidence < self.CONFIDENCE_FLOOR:
            reasons.append(
                f"Policy gap: confidence {confidence:.2f} is below "
                f"{self.CONFIDENCE_FLOOR:.2f} floor"
            )
            triggered.append("POLICY_GAP")
            max_priority = max(max_priority, EscalationPriority.MEDIUM,
                               key=lambda p: p.value)

        # --- Rule 5: Repeated failures ---
        attempts = request.get("prior_attempts", 0)
        if attempts >= self.MAX_FAILED_ATTEMPTS:
            reasons.append(
                f"Repeated failures: {attempts} prior attempts "
                f"(threshold: {self.MAX_FAILED_ATTEMPTS})"
            )
            triggered.append("REPEATED_FAILURES")
            max_priority = max(max_priority, EscalationPriority.MEDIUM,
                               key=lambda p: p.value)

        should_escalate = max_priority.value >= EscalationPriority.MEDIUM.value
        classifier_confidence = 0.95 if len(triggered) > 0 else 0.85

        return EscalationResult(
            should_escalate=should_escalate,
            priority=max_priority,
            reasons=reasons,
            triggered_rules=triggered,
            confidence=classifier_confidence,
        )


# Instantiate
classifier = EscalationClassifier()

print("EscalationClassifier ready.")
print(f"  Safety keywords:        {len(classifier.SAFETY_KEYWORDS)}")
print(f"  Human request keywords: {len(classifier.HUMAN_REQUEST_KEYWORDS)}")
print(f"  High-value threshold:   ${classifier.HIGH_VALUE_THRESHOLD:,}")
print(f"  Confidence floor:       {classifier.CONFIDENCE_FLOOR}")
print(f"  Max failed attempts:    {classifier.MAX_FAILED_ATTEMPTS}")

In [ ]:
# ============================================================
# 3B: Run the classifier on our 10 test messages
# ============================================================

# Enrich the test messages with the additional fields the classifier needs
enriched_messages = [
    {**msg, "prior_attempts": 0, "policy_match_confidence": 0.95}
    for msg in test_messages
]

# Override specific fields for messages that need them
enriched_messages[4]["prior_attempts"] = 3      # Message #5: third time calling
enriched_messages[8]["policy_match_confidence"] = 0.35  # Message #9: policy gap

print("CLASSIFIER RESULTS:")
print("=" * 70)

for req in enriched_messages:
    result = classifier.classify(req)
    status = "ESCALATE" if result.should_escalate else "HANDLE"
    priority_str = result.priority.name
    print(f"\n  #{req['id']}: \"{req['message'][:50]}...\"")
    print(f"      Decision: {status} | Priority: {priority_str}")
    if result.triggered_rules:
        print(f"      Rules triggered: {result.triggered_rules}")
    for reason in result.reasons:
        print(f"      - {reason}")

In [ ]:
# ============================================================
# 3C: Compare classifier decisions to the three team members
# ============================================================

# Get classifier decisions
classifier_decisions = {}
for req in enriched_messages:
    result = classifier.classify(req)
    classifier_decisions[req["id"]] = "ESCALATE" if result.should_escalate else "HANDLE"

print("CONSISTENCY COMPARISON: Humans vs Classifier")
print("=" * 75)
print(f"\n  {'Msg':<5} {'Alice':<12} {'Bob':<12} {'Carol':<12} {'Classifier':<12} {'Consistent?'}")
print(f"  {'-'*5} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")

for msg_id in range(1, 11):
    a = team_decisions["Alice (cautious)"][msg_id]
    b = team_decisions["Bob (aggressive)"][msg_id]
    c = team_decisions["Carol (moderate)"][msg_id]
    cl = classifier_decisions[msg_id]

    # "Consistent" means the classifier agrees with the majority
    human_votes = [a, b, c]
    majority = max(set(human_votes), key=human_votes.count)
    consistent = "YES" if cl == majority else "DIFFERS"

    print(f"  #{msg_id:<4} {a:<12} {b:<12} {c:<12} {cl:<12} {consistent}")

print(f"\n  Key advantage: The classifier gives the SAME answer every time.")
print(f"  No variation by time of day, mood, or interpretation.")
print(f"  And every decision is auditable -- you can trace exactly which")
print(f"  rule fired and why.")

### What We Built

The classifier evaluates each request against **5 concrete rules** with **measurable thresholds**:

| Rule | Threshold | Priority |
|---|---|---|
| Safety concern | Keyword match from safety list | CRITICAL |
| Customer requests human | Keyword match from human-request list | HIGH |
| High-value decision | Claim > $100,000 | HIGH |
| Policy gap | Confidence < 0.70 | MEDIUM |
| Repeated failures | 3+ prior failed attempts | MEDIUM |

Every decision is traceable: you can see exactly which rules fired, what the thresholds were, and why the classifier made its call. This is auditable in a way that "I felt like this was complex" never will be.

---

In [ ]:
#@title 🎧 Listen: Handoff Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_handoff_todo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Handoff Scorer Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_handoff_scorer_todo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 4: Exercise -- Structured Handoff Protocol

When the classifier decides to escalate, the agent must hand off to a human. But a bare escalation -- "I could not handle this" -- forces the human to start from scratch. A **structured handoff summary** preserves everything the agent learned so the human can pick up mid-conversation.

In [ ]:
# ============================================================
# 4A: A bad handoff vs a good handoff
# ============================================================

bad_handoff = {
    "status": "escalated",
    "reason": "Could not resolve customer issue."
}

good_handoff = {
    "escalation_reason": "policy_gap",
    "triggered_rules": ["POLICY_GAP"],
    "confidence_score": 0.35,
    "customer_id": "CUST-44891",
    "conversation_summary": (
        "Customer claims water damage to basement. "
        "Policy covers flood damage but does not define whether "
        "groundwater seepage qualifies as 'flood.'"
    ),
    "what_was_attempted": [
        "Searched policy document for 'groundwater' -- no matches",
        "Searched policy document for 'seepage' -- no matches",
        "Searched FAQ database for 'groundwater coverage' -- no entries",
        "Checked claim precedent database -- no similar cases found",
    ],
    "what_remains": [
        "Determine if groundwater seepage falls under flood coverage",
        "Check company precedent for similar claims",
        "Consult underwriting team if no precedent exists",
    ],
    "relevant_context": {
        "policy_number": "HO-2024-33892",
        "claim_amount": 28500,
        "policy_sections_reviewed": [
            "Section 4.2 -- Water Damage",
            "Section 4.3 -- Flood Coverage",
            "Section 8.1 -- Exclusions",
        ],
    },
    "customer_sentiment": "frustrated but cooperative",
    "suggested_action": (
        "Human reviewer should determine if groundwater seepage "
        "falls under flood coverage per company precedent."
    ),
    "escalation_timestamp": "2024-03-15T14:32:00Z",
}

print("BAD HANDOFF:")
print("-" * 40)
pp(bad_handoff)

print("\n\nGOOD HANDOFF:")
print("-" * 40)
pp(good_handoff)

print("\n\nThe human reviewer gets:")
print("  - WHY it was escalated (policy gap, confidence 0.35)")
print("  - WHAT was already tried (4 search attempts)")
print("  - WHAT remains to be done (3 specific next steps)")
print("  - ALL relevant context (policy number, claim amount, sections)")
print("  - A SUGGESTED action (specific recommendation)")
print("\nThey can start working immediately instead of re-reading the whole case.")

In [ ]:
# ============================================================
# 4B: TODO -- Build the handoff generator
# ============================================================

# TODO: Complete the generate_handoff function below.
# It should take a request dict and an EscalationResult, and produce
# a structured handoff summary like the good_handoff example above.

def generate_handoff(request: dict, escalation: EscalationResult) -> dict:
    """
    Generate a structured handoff summary for human review.

    Args:
        request: The original request dict with message, claim_value,
                 context, etc.
        escalation: The EscalationResult from the classifier.

    Returns:
        A structured handoff dict with all fields needed for
        a human to pick up the case without starting over.
    """

    # TODO 1: Fill in the escalation_reason field.
    # Use the first triggered rule, or "unknown" if none triggered.
    escalation_reason = "TODO"  # <-- Replace this

    # TODO 2: Build the conversation_summary.
    # Combine the customer message and the context into a clear summary.
    conversation_summary = "TODO"  # <-- Replace this

    # TODO 3: Build the what_was_attempted list.
    # In a real system, this would come from the agent's action log.
    # For this exercise, simulate 2-3 reasonable search attempts
    # based on the customer's message.
    what_was_attempted = []  # <-- Fill this in

    # TODO 4: Build the what_remains list.
    # What specific actions does the human need to take?
    what_remains = []  # <-- Fill this in

    # TODO 5: Add a suggested_action string.
    # A specific, actionable recommendation for the human reviewer.
    suggested_action = "TODO"  # <-- Replace this

    handoff = {
        "escalation_reason": escalation_reason,
        "triggered_rules": escalation.triggered_rules,
        "confidence_score": escalation.confidence,
        "customer_message": request.get("message", ""),
        "claim_value": request.get("claim_value", 0),
        "conversation_summary": conversation_summary,
        "what_was_attempted": what_was_attempted,
        "what_remains": what_remains,
        "relevant_context": {
            "prior_attempts": request.get("prior_attempts", 0),
            "policy_match_confidence": request.get(
                "policy_match_confidence", None
            ),
        },
        "suggested_action": suggested_action,
        "escalation_timestamp": datetime.now().isoformat(),
    }

    return handoff


# Test your implementation
test_request = enriched_messages[8]  # Message #9: groundwater seepage
test_escalation = classifier.classify(test_request)

handoff = generate_handoff(test_request, test_escalation)
print("YOUR HANDOFF:")
print("=" * 60)
pp(handoff)

In [ ]:
# ============================================================
# 4C: SOLUTION -- Handoff generator
# ============================================================

def generate_handoff_solution(request: dict, escalation: EscalationResult) -> dict:
    """
    Generate a structured handoff summary for human review.
    """
    # 1. Escalation reason from the first triggered rule
    escalation_reason = (
        escalation.triggered_rules[0].lower()
        if escalation.triggered_rules
        else "unknown"
    )

    # 2. Conversation summary from message + context
    conversation_summary = (
        f"Customer message: \"{request.get('message', '')}\". "
        f"Context: {request.get('context', 'No additional context.')} "
        f"Claim value: ${request.get('claim_value', 0):,.2f}."
    )

    # 3. What was attempted (simulated based on common agent actions)
    message = request.get("message", "").lower()
    what_was_attempted = [
        "Searched policy documents for relevant coverage terms",
        "Checked FAQ database for matching questions",
    ]
    if request.get("claim_value", 0) > 0:
        what_was_attempted.append(
            "Verified claim amount against policy coverage limits"
        )
    if request.get("prior_attempts", 0) > 0:
        what_was_attempted.append(
            f"Reviewed {request['prior_attempts']} prior interaction(s) "
            f"for resolution attempts"
        )
    if request.get("policy_match_confidence", 1.0) < 0.7:
        what_was_attempted.append(
            "Searched for policy precedent -- no matching cases found"
        )

    # 4. What remains
    what_remains = []
    if "policy_gap" in escalation_reason:
        what_remains.append(
            "Determine applicable policy interpretation for this scenario"
        )
        what_remains.append("Check company precedent database")
    if "safety" in escalation_reason:
        what_remains.append("Assess safety situation and respond appropriately")
        what_remains.append("Document incident per compliance requirements")
    if "high_value" in escalation_reason:
        what_remains.append("Senior review required for high-value decision")
    if request.get("prior_attempts", 0) >= 3:
        what_remains.append(
            "Investigate why prior resolution attempts failed"
        )
    if not what_remains:
        what_remains.append("Human reviewer to assess and resolve directly")

    # 5. Suggested action
    priority_actions = {
        "safety_concern": (
            "Immediate human attention required. Assess the safety "
            "situation and follow incident response protocol."
        ),
        "customer_requests_human": (
            "Connect customer to a live agent promptly. "
            "Customer has explicitly requested human assistance."
        ),
        "high_value_decision": (
            "Route to senior reviewer for authorization. "
            f"Decision involves ${request.get('claim_value', 0):,.2f}."
        ),
        "policy_gap": (
            "Consult policy team to determine coverage for this "
            "scenario. No existing precedent was found."
        ),
        "repeated_failures": (
            "Investigate root cause of repeated failures. "
            "Customer has contacted support multiple times without resolution."
        ),
    }
    suggested_action = priority_actions.get(
        escalation_reason,
        "Human reviewer to assess the situation and determine next steps."
    )

    return {
        "escalation_reason": escalation_reason,
        "triggered_rules": escalation.triggered_rules,
        "priority": escalation.priority.name,
        "confidence_score": escalation.confidence,
        "customer_message": request.get("message", ""),
        "claim_value": request.get("claim_value", 0),
        "conversation_summary": conversation_summary,
        "what_was_attempted": what_was_attempted,
        "what_remains": what_remains,
        "relevant_context": {
            "prior_attempts": request.get("prior_attempts", 0),
            "policy_match_confidence": request.get(
                "policy_match_confidence", None
            ),
        },
        "suggested_action": suggested_action,
        "escalation_timestamp": datetime.now().isoformat(),
    }


# Test on all messages that trigger escalation
print("STRUCTURED HANDOFFS FOR ALL ESCALATED MESSAGES:")
print("=" * 70)
for req in enriched_messages:
    result = classifier.classify(req)
    if result.should_escalate:
        handoff = generate_handoff_solution(req, result)
        print(f"\n--- Message #{req['id']}: Priority {handoff['priority']} ---")
        pp(handoff)

In [ ]:
# ============================================================
# 4D: TODO -- Evaluate handoff quality
# ============================================================

# A good handoff summary should have all of these components.
# TODO: Complete the score_handoff function to check each one.

def score_handoff(handoff: dict) -> dict:
    """
    Score a handoff summary on completeness.
    Returns a dict with scores for each component (0 or 1)
    and an overall percentage.
    """
    scores = {}

    # TODO 1: Check if escalation_reason is present and not "TODO"/"unknown"
    scores["has_reason"] = 0  # <-- Replace with your check

    # TODO 2: Check if conversation_summary is present and >20 chars
    scores["has_summary"] = 0  # <-- Replace with your check

    # TODO 3: Check if what_was_attempted has at least 2 entries
    scores["has_attempts"] = 0  # <-- Replace with your check

    # TODO 4: Check if what_remains has at least 1 entry
    scores["has_remaining"] = 0  # <-- Replace with your check

    # TODO 5: Check if suggested_action is present and not "TODO"
    scores["has_suggestion"] = 0  # <-- Replace with your check

    total = sum(scores.values())
    max_score = len(scores)
    scores["percentage"] = round(total / max_score * 100, 1)

    return scores


# Test on the bad handoff and a good one
print("Scoring BAD handoff:")
bad_as_handoff = {
    "escalation_reason": "unknown",
    "conversation_summary": "",
    "what_was_attempted": [],
    "what_remains": [],
    "suggested_action": "",
}
pp(score_handoff(bad_as_handoff))

print("\nScoring GOOD handoff:")
good_test = generate_handoff_solution(enriched_messages[8],
                                       classifier.classify(enriched_messages[8]))
pp(score_handoff(good_test))

In [ ]:
# ============================================================
# 4E: SOLUTION -- Handoff quality scorer
# ============================================================

def score_handoff_solution(handoff: dict) -> dict:
    """Score a handoff summary on completeness."""
    scores = {}

    reason = handoff.get("escalation_reason", "")
    scores["has_reason"] = int(
        bool(reason) and reason not in ("TODO", "unknown", "")
    )

    summary = handoff.get("conversation_summary", "")
    scores["has_summary"] = int(len(summary) > 20)

    attempts = handoff.get("what_was_attempted", [])
    scores["has_attempts"] = int(len(attempts) >= 2)

    remaining = handoff.get("what_remains", [])
    scores["has_remaining"] = int(len(remaining) >= 1)

    suggestion = handoff.get("suggested_action", "")
    scores["has_suggestion"] = int(
        bool(suggestion) and suggestion != "TODO"
    )

    total = sum(scores.values())
    max_score = len(scores)
    scores["percentage"] = round(total / max_score * 100, 1)

    return scores


# Compare
print("BAD handoff score:")
pp(score_handoff_solution(bad_as_handoff))

print("\nGOOD handoff score:")
pp(score_handoff_solution(good_test))

print("\nA production system should require >= 80% handoff completeness")
print("before allowing the escalation to enter the human review queue.")

In [ ]:
#@title 🎧 Listen: Stratified Sampling
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_stratified_sampling.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Category Classifier Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_category_classifier_todo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 5: Exercise -- Stratified Sampling for Quality Review

Even when the agent handles requests without escalating, some of those decisions may be wrong. **Human review** catches these errors -- but reviewing every decision is impossible at scale. The question is: which decisions do you review?

In [ ]:
# ============================================================
# 5A: The problem with random sampling
# ============================================================

# Simulate 10,000 agent decisions across different categories
random.seed(42)

categories = {
    "simple_auto_approved": {
        "count": 7500,
        "actual_error_rate": 0.01,  # 1% error rate
        "description": "Routine requests (status checks, address updates)"
    },
    "medium_complexity": {
        "count": 1500,
        "actual_error_rate": 0.08,  # 8% error rate
        "description": "Policy questions, moderate claims"
    },
    "high_value": {
        "count": 600,
        "actual_error_rate": 0.15,  # 15% error rate
        "description": "Claims between $10K-$100K"
    },
    "edge_cases": {
        "count": 300,
        "actual_error_rate": 0.30,  # 30% error rate
        "description": "Unusual situations, policy ambiguities"
    },
    "new_policy_types": {
        "count": 100,
        "actual_error_rate": 0.45,  # 45% error rate
        "description": "Recently added coverage types"
    },
}

# Generate simulated decisions
all_decisions = []
for cat_name, cat_info in categories.items():
    for _ in range(cat_info["count"]):
        is_error = random.random() < cat_info["actual_error_rate"]
        all_decisions.append({
            "category": cat_name,
            "has_error": is_error,
            "claim_value": random.uniform(50, 300000)
            if cat_name == "high_value"
            else random.uniform(10, 5000),
        })

total_errors = sum(1 for d in all_decisions if d["has_error"])
print(f"Simulated {len(all_decisions):,} agent decisions")
print(f"Total actual errors: {total_errors}")
print(f"Overall error rate: {total_errors/len(all_decisions):.1%}")

print("\nError distribution by category:")
print("-" * 60)
for cat_name, cat_info in categories.items():
    cat_decisions = [d for d in all_decisions if d["category"] == cat_name]
    cat_errors = sum(1 for d in cat_decisions if d["has_error"])
    print(f"  {cat_name:25s}: {cat_errors:4d} errors in "
          f"{cat_info['count']:5d} decisions "
          f"({cat_errors/cat_info['count']:.1%})")

In [ ]:
# ============================================================
# 5B: Random sampling -- how many errors do we catch?
# ============================================================

REVIEW_BUDGET = 200  # We can review 200 decisions

# Random sampling: pick 200 decisions at random
random.seed(42)
random_sample = random.sample(all_decisions, REVIEW_BUDGET)

random_errors_found = sum(1 for d in random_sample if d["has_error"])
random_category_breakdown = defaultdict(lambda: {"reviewed": 0, "errors": 0})

for d in random_sample:
    random_category_breakdown[d["category"]]["reviewed"] += 1
    if d["has_error"]:
        random_category_breakdown[d["category"]]["errors"] += 1

print(f"RANDOM SAMPLING: {REVIEW_BUDGET} reviews")
print("=" * 60)
print(f"\n  Total errors found: {random_errors_found}")
print(f"  Error detection rate: {random_errors_found}/{total_errors} "
      f"({random_errors_found/total_errors:.1%})")

print(f"\n  Category breakdown:")
print(f"  {'Category':<25} {'Reviewed':<10} {'Errors Found'}")
print(f"  {'-'*25} {'-'*10} {'-'*12}")
for cat_name in categories:
    info = random_category_breakdown[cat_name]
    print(f"  {cat_name:<25} {info['reviewed']:<10} {info['errors']}")

print(f"\n  Problem: Random sampling reviews categories in proportion to")
print(f"  their VOLUME, not their ERROR RATE. You spend most of your")
print(f"  budget reviewing simple cases that are almost always correct.")

In [ ]:
# ============================================================
# 5C: Stratified sampling -- target high-risk categories
# ============================================================

def stratified_sample(decisions: list, budget: int) -> list:
    """
    Sample proportionally to error risk, not volume.

    Higher sample rates for categories with higher expected error rates.
    """
    # Define sample rates per category (higher = more review)
    sample_rates = {
        "simple_auto_approved": 0.005,  # 0.5% -- very low risk
        "medium_complexity":    0.03,   # 3%
        "high_value":           0.15,   # 15%
        "edge_cases":           0.30,   # 30%
        "new_policy_types":     0.60,   # 60% -- highest risk
    }

    # Group decisions by category
    by_category = defaultdict(list)
    for d in decisions:
        by_category[d["category"]].append(d)

    # Sample from each category at its designated rate
    review_queue = []
    for cat_name, cat_decisions in by_category.items():
        rate = sample_rates.get(cat_name, 0.05)
        n_to_sample = max(1, int(len(cat_decisions) * rate))
        n_to_sample = min(n_to_sample, len(cat_decisions))
        sampled = random.sample(cat_decisions, n_to_sample)
        for d in sampled:
            review_queue.append({
                **d,
                "sample_rate": rate,
                "priority": rate,  # higher rate = higher review priority
            })

    # Sort by priority (highest-risk first) and trim to budget
    review_queue.sort(key=lambda x: -x["priority"])
    return review_queue[:budget]


random.seed(42)
stratified = stratified_sample(all_decisions, REVIEW_BUDGET)

strat_errors_found = sum(1 for d in stratified if d["has_error"])
strat_category_breakdown = defaultdict(lambda: {"reviewed": 0, "errors": 0})

for d in stratified:
    strat_category_breakdown[d["category"]]["reviewed"] += 1
    if d["has_error"]:
        strat_category_breakdown[d["category"]]["errors"] += 1

print(f"STRATIFIED SAMPLING: {REVIEW_BUDGET} reviews")
print("=" * 60)
print(f"\n  Total errors found: {strat_errors_found}")
print(f"  Error detection rate: {strat_errors_found}/{total_errors} "
      f"({strat_errors_found/total_errors:.1%})")

print(f"\n  Category breakdown:")
print(f"  {'Category':<25} {'Reviewed':<10} {'Errors Found'}")
print(f"  {'-'*25} {'-'*10} {'-'*12}")
for cat_name in categories:
    info = strat_category_breakdown[cat_name]
    print(f"  {cat_name:<25} {info['reviewed']:<10} {info['errors']}")

print(f"\n  Improvement: Stratified sampling found {strat_errors_found} errors")
print(f"  vs {random_errors_found} with random sampling.")
print(f"  That is {strat_errors_found - random_errors_found} more errors caught")
print(f"  with the SAME review budget of {REVIEW_BUDGET}.")

In [ ]:
# ============================================================
# 5D: TODO -- Build the category classifier
# ============================================================

# TODO: Implement classify_decision_complexity to sort each decision
# into one of the 5 risk categories. The classifier should use
# concrete, measurable criteria -- not vague heuristics.

def classify_decision_complexity(decision: dict) -> str:
    """
    Classify a decision into a risk category for stratified sampling.

    Args:
        decision: dict with keys:
            - claim_value (float): dollar amount
            - message (str): customer message
            - prior_attempts (int): number of prior contacts
            - days_since_policy_start (int): how new the policy is
            - policy_match_confidence (float): 0-1

    Returns:
        Category name as string.
    """
    # TODO 1: Classify as "new_policy_types" if the policy started
    # within the last 90 days.

    # TODO 2: Classify as "edge_cases" if policy_match_confidence < 0.6
    # OR if prior_attempts >= 2.

    # TODO 3: Classify as "high_value" if claim_value > $10,000.

    # TODO 4: Classify as "medium_complexity" if claim_value > $500
    # OR policy_match_confidence < 0.85.

    # TODO 5: Default to "simple_auto_approved" for everything else.

    return "simple_auto_approved"  # <-- Replace with your logic


# Test your classifier
test_cases = [
    {"claim_value": 50, "prior_attempts": 0,
     "days_since_policy_start": 365, "policy_match_confidence": 0.98,
     "message": "Check order status"},
    {"claim_value": 150000, "prior_attempts": 0,
     "days_since_policy_start": 200, "policy_match_confidence": 0.90,
     "message": "File major claim"},
    {"claim_value": 1000, "prior_attempts": 0,
     "days_since_policy_start": 30, "policy_match_confidence": 0.80,
     "message": "New policy question"},
    {"claim_value": 500, "prior_attempts": 3,
     "days_since_policy_start": 400, "policy_match_confidence": 0.50,
     "message": "Third time calling about same issue"},
]

print("YOUR CLASSIFIER RESULTS:")
for tc in test_cases:
    result = classify_decision_complexity(tc)
    print(f"  ${tc['claim_value']:>10,.2f} | conf={tc['policy_match_confidence']:.2f} "
          f"| attempts={tc['prior_attempts']} | -> {result}")

In [ ]:
# ============================================================
# 5E: SOLUTION -- Category classifier
# ============================================================

def classify_decision_complexity_solution(decision: dict) -> str:
    """Classify a decision into a risk category for stratified sampling."""
    claim_value = decision.get("claim_value", 0)
    confidence = decision.get("policy_match_confidence", 1.0)
    attempts = decision.get("prior_attempts", 0)
    days = decision.get("days_since_policy_start", 999)

    # Priority ordering matters -- check highest risk first
    if days < 90:
        return "new_policy_types"

    if confidence < 0.6 or attempts >= 2:
        return "edge_cases"

    if claim_value > 10000:
        return "high_value"

    if claim_value > 500 or confidence < 0.85:
        return "medium_complexity"

    return "simple_auto_approved"


# Test the solution
print("SOLUTION CLASSIFIER RESULTS:")
for tc in test_cases:
    result = classify_decision_complexity_solution(tc)
    print(f"  ${tc['claim_value']:>10,.2f} | conf={tc['policy_match_confidence']:.2f} "
          f"| attempts={tc['prior_attempts']} | days={tc['days_since_policy_start']:>3} "
          f"| -> {result}")

print("\nNote: The order of checks matters! We check the highest-risk")
print("categories first (new_policy_types, edge_cases) so that a new")
print("policy with a high claim value is classified as 'new_policy_types'")
print("rather than 'high_value' -- because the newness is the bigger risk.")

In [ ]:
#@title 🎧 Listen: Adaptive Rates
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_adaptive_rates.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 6: Exercise -- Adapting Sample Rates Over Time

In a real system, you do not set sample rates once and forget them. You **track per-category accuracy** and adjust sample rates based on observed performance.

In [ ]:
# ============================================================
# 6A: Accuracy tracking over time
# ============================================================

# Simulate 4 weeks of review data
weekly_accuracy = {
    "week_1": {
        "simple_auto_approved": {"reviewed": 38, "correct": 37},
        "medium_complexity":    {"reviewed": 45, "correct": 41},
        "high_value":           {"reviewed": 60, "correct": 51},
        "edge_cases":           {"reviewed": 40, "correct": 28},
        "new_policy_types":     {"reviewed": 17, "correct": 8},
    },
    "week_2": {
        "simple_auto_approved": {"reviewed": 40, "correct": 39},
        "medium_complexity":    {"reviewed": 42, "correct": 39},
        "high_value":           {"reviewed": 55, "correct": 48},
        "edge_cases":           {"reviewed": 45, "correct": 33},
        "new_policy_types":     {"reviewed": 18, "correct": 10},
    },
    "week_3": {
        "simple_auto_approved": {"reviewed": 35, "correct": 35},
        "medium_complexity":    {"reviewed": 48, "correct": 45},
        "high_value":           {"reviewed": 50, "correct": 44},
        "edge_cases":           {"reviewed": 42, "correct": 34},
        "new_policy_types":     {"reviewed": 25, "correct": 16},
    },
    "week_4": {
        "simple_auto_approved": {"reviewed": 37, "correct": 37},
        "medium_complexity":    {"reviewed": 44, "correct": 42},
        "high_value":           {"reviewed": 58, "correct": 52},
        "edge_cases":           {"reviewed": 38, "correct": 31},
        "new_policy_types":     {"reviewed": 23, "correct": 17},
    },
}

print("WEEKLY ACCURACY BY CATEGORY:")
print("=" * 70)
for week, data in weekly_accuracy.items():
    print(f"\n  {week}:")
    for cat, stats in data.items():
        rate = stats["correct"] / stats["reviewed"] if stats["reviewed"] > 0 else 0
        bar = "#" * int(rate * 20) + "." * (20 - int(rate * 20))
        print(f"    {cat:25s} [{bar}] {rate:.1%}")

In [ ]:
# ============================================================
# 6B: TODO -- Adaptive sample rate calculator
# ============================================================

# TODO: Implement a function that adjusts sample rates based on
# observed accuracy. Categories with LOWER accuracy should get
# HIGHER sample rates in the next period.

def calculate_adaptive_rates(
    accuracy_history: dict,
    base_rates: dict,
    min_rate: float = 0.005,
    max_rate: float = 0.80,
) -> dict:
    """
    Adjust sample rates based on observed accuracy.

    Args:
        accuracy_history: dict mapping category -> {reviewed, correct}
        base_rates: dict mapping category -> current sample rate
        min_rate: minimum allowed sample rate
        max_rate: maximum allowed sample rate

    Returns:
        dict mapping category -> new sample rate
    """
    new_rates = {}

    for category, stats in accuracy_history.items():
        # TODO: Calculate the accuracy rate for this category
        accuracy = 0.0  # <-- Replace with actual calculation

        # TODO: Calculate the new sample rate.
        # The idea: lower accuracy -> higher sample rate.
        # One approach: new_rate = base_rate * (1 / accuracy)
        # But clamp between min_rate and max_rate.
        new_rate = base_rates.get(category, 0.05)  # <-- Replace

        # Clamp to bounds
        new_rates[category] = max(min_rate, min(max_rate, new_rate))

    return new_rates


# Test with week 4 data
base_rates = {
    "simple_auto_approved": 0.005,
    "medium_complexity":    0.03,
    "high_value":           0.15,
    "edge_cases":           0.30,
    "new_policy_types":     0.60,
}

new_rates = calculate_adaptive_rates(
    weekly_accuracy["week_4"], base_rates
)

print("ADAPTIVE SAMPLE RATES:")
print("=" * 60)
for cat in base_rates:
    print(f"  {cat:25s}: {base_rates[cat]:.3f} -> {new_rates[cat]:.3f}")

In [ ]:
# ============================================================
# 6C: SOLUTION -- Adaptive sample rate calculator
# ============================================================

def calculate_adaptive_rates_solution(
    accuracy_history: dict,
    base_rates: dict,
    min_rate: float = 0.005,
    max_rate: float = 0.80,
) -> dict:
    """Adjust sample rates based on observed accuracy."""
    new_rates = {}

    for category, stats in accuracy_history.items():
        reviewed = stats.get("reviewed", 0)
        correct = stats.get("correct", 0)

        if reviewed == 0:
            # No data -- keep the base rate
            new_rates[category] = base_rates.get(category, 0.05)
            continue

        accuracy = correct / reviewed

        # Core formula: invert accuracy to get risk multiplier
        # Accuracy 1.0 -> multiplier 1.0 (keep same rate)
        # Accuracy 0.5 -> multiplier 2.0 (double the rate)
        # Accuracy 0.25 -> multiplier 4.0 (quadruple the rate)
        if accuracy > 0:
            risk_multiplier = 1.0 / accuracy
        else:
            risk_multiplier = 10.0  # Max boost for zero accuracy

        # Apply multiplier to base rate
        base = base_rates.get(category, 0.05)
        new_rate = base * risk_multiplier

        # Smooth the change: blend 70% new + 30% old to avoid
        # wild swings from noisy data
        smoothed = 0.7 * new_rate + 0.3 * base
        new_rates[category] = max(min_rate, min(max_rate, smoothed))

    return new_rates


# Run solution
new_rates_solution = calculate_adaptive_rates_solution(
    weekly_accuracy["week_4"], base_rates
)

print("SOLUTION -- ADAPTIVE SAMPLE RATES:")
print("=" * 65)
print(f"  {'Category':<25} {'Base Rate':>10} {'New Rate':>10} {'Change':>10}")
print(f"  {'-'*25} {'-'*10} {'-'*10} {'-'*10}")
for cat in base_rates:
    old = base_rates[cat]
    new = new_rates_solution[cat]
    change = (new - old) / old * 100
    direction = "+" if change > 0 else ""
    print(f"  {cat:<25} {old:>10.3f} {new:>10.3f} {direction}{change:>8.1f}%")

print("\nCategories with lower accuracy get higher sample rates.")
print("'new_policy_types' (74% accuracy) gets a rate boost.")
print("'simple_auto_approved' (100% accuracy) stays minimal.")

In [ ]:
#@title 🎧 Listen: Visualizations
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_visualizations.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 7: Visualization -- Stratified vs Random Sampling

Let's make the comparison visual.

In [ ]:
# ============================================================
# 7A: Side-by-side comparison chart
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# -- Left: Errors found by each method --
ax = axes[0]
methods = ["Random\nSampling", "Stratified\nSampling"]
errors_found = [random_errors_found, strat_errors_found]
colors = ["#ff6b6b", "#51cf66"]

bars = ax.bar(methods, errors_found, color=colors, edgecolor="white",
              linewidth=2, width=0.5)
for bar, count in zip(bars, errors_found):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha="center", fontsize=16, fontweight="bold")

ax.set_ylabel("Errors Found", fontsize=13)
ax.set_title(f"Errors Detected\n(Review Budget: {REVIEW_BUDGET})",
             fontsize=14, fontweight="bold")
ax.set_ylim(0, max(errors_found) * 1.25)
ax.axhline(y=total_errors, color="gray", linestyle="--", alpha=0.5)
ax.text(1.3, total_errors + 1, f"Total errors: {total_errors}",
        fontsize=10, color="gray")

# -- Right: Category distribution of reviews --
ax = axes[1]
cat_names_short = ["Simple", "Medium", "High\nValue", "Edge\nCases", "New\nPolicy"]
cat_keys = list(categories.keys())

random_counts = [random_category_breakdown[k]["reviewed"] for k in cat_keys]
strat_counts = [strat_category_breakdown[k]["reviewed"] for k in cat_keys]

x = np.arange(len(cat_names_short))
width = 0.35

bars1 = ax.bar(x - width / 2, random_counts, width, label="Random",
               color="#ff6b6b", alpha=0.85)
bars2 = ax.bar(x + width / 2, strat_counts, width, label="Stratified",
               color="#51cf66", alpha=0.85)

ax.set_ylabel("Reviews Allocated", fontsize=13)
ax.set_title("Where Reviews Are Spent\nby Category", fontsize=14,
             fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(cat_names_short, fontsize=10)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Left chart: Stratified sampling finds more errors with the same budget.")
print("Right chart: Stratified spending shifts reviews to high-risk categories,")
print("while random sampling wastes most of its budget on simple cases.")

In [ ]:
# ============================================================
# 7B: Per-category error detection rate comparison
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

# For each category, calculate what percentage of errors were found
cat_names_short = ["Simple", "Medium", "High Value", "Edge Cases", "New Policy"]
cat_keys = list(categories.keys())

random_detection = []
strat_detection = []

for k in cat_keys:
    cat_total_errors = sum(
        1 for d in all_decisions
        if d["category"] == k and d["has_error"]
    )
    if cat_total_errors == 0:
        random_detection.append(0)
        strat_detection.append(0)
    else:
        r_found = random_category_breakdown[k]["errors"]
        s_found = strat_category_breakdown[k]["errors"]
        random_detection.append(r_found / cat_total_errors * 100)
        strat_detection.append(s_found / cat_total_errors * 100)

x = np.arange(len(cat_names_short))
width = 0.35

ax.bar(x - width / 2, random_detection, width, label="Random Sampling",
       color="#ff6b6b", alpha=0.85)
ax.bar(x + width / 2, strat_detection, width, label="Stratified Sampling",
       color="#51cf66", alpha=0.85)

ax.set_ylabel("Error Detection Rate (%)", fontsize=13)
ax.set_xlabel("Category", fontsize=13)
ax.set_title("Error Detection Rate by Category\n(What % of actual errors were caught?)",
             fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(cat_names_short, fontsize=11)
ax.legend(fontsize=11, loc="upper left")
ax.set_ylim(0, 105)

# Add a horizontal line at 100%
ax.axhline(y=100, color="gray", linestyle=":", alpha=0.3)

plt.tight_layout()
plt.show()

print("Stratified sampling achieves higher detection rates in EVERY high-risk category.")
print("For 'New Policy Types' -- the most error-prone category -- stratified sampling")
print("reviews 60% of cases vs ~2% with random sampling.")

In [ ]:
# ============================================================
# 7C: Accuracy trend over 4 weeks
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

weeks = ["Week 1", "Week 2", "Week 3", "Week 4"]
colors_map = {
    "simple_auto_approved": "#51cf66",
    "medium_complexity": "#74c0fc",
    "high_value": "#ffd43b",
    "edge_cases": "#ff922b",
    "new_policy_types": "#ff6b6b",
}
labels_map = {
    "simple_auto_approved": "Simple",
    "medium_complexity": "Medium",
    "high_value": "High Value",
    "edge_cases": "Edge Cases",
    "new_policy_types": "New Policy",
}

for cat_name in categories:
    rates = []
    for week_key in ["week_1", "week_2", "week_3", "week_4"]:
        stats = weekly_accuracy[week_key][cat_name]
        rate = stats["correct"] / stats["reviewed"] if stats["reviewed"] > 0 else 0
        rates.append(rate * 100)
    ax.plot(weeks, rates, "o-", linewidth=2.5, markersize=8,
            color=colors_map[cat_name], label=labels_map[cat_name])

ax.set_ylabel("Accuracy (%)", fontsize=13)
ax.set_xlabel("Review Period", fontsize=13)
ax.set_title("Agent Accuracy by Category Over Time",
             fontsize=14, fontweight="bold")
ax.set_ylim(40, 105)
ax.legend(fontsize=11, loc="lower right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("This chart tells you exactly where to invest improvement effort:")
print("  - 'New Policy Types' accuracy is improving (47% -> 74%)")
print("    but still needs work.")
print("  - 'Simple' cases are near-perfect -- reduce review budget here.")
print("  - 'Edge Cases' hover around 70-80% -- needs prompt engineering.")

In [ ]:
#@title 🎧 Listen: Mini Project
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_mini_project.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 8: Mini-Project -- Complete Escalation Pipeline

Now let's bring everything together: classifier, handoff generator, and stratified sampling into one integrated pipeline.

In [ ]:
# ============================================================
# 8A: The integrated pipeline
# ============================================================

class EscalationPipeline:
    """
    End-to-end pipeline: classify -> handle or escalate -> sample for review.
    """

    def __init__(self):
        self.classifier = EscalationClassifier()
        self.handled = []
        self.escalated = []
        self.review_queue = []
        self.stats = defaultdict(lambda: {"total": 0, "escalated": 0,
                                           "handled": 0, "errors_found": 0})

    def process_request(self, request: dict) -> dict:
        """Process a single request through the full pipeline."""
        # Step 1: Classify
        escalation = self.classifier.classify(request)

        if escalation.should_escalate:
            # Step 2a: Generate handoff and escalate
            handoff = generate_handoff_solution(request, escalation)
            self.escalated.append(handoff)
            self.stats[request.get("category", "unknown")]["escalated"] += 1
            outcome = {
                "action": "ESCALATED",
                "priority": escalation.priority.name,
                "handoff": handoff,
            }
        else:
            # Step 2b: Agent handles it
            self.handled.append(request)
            self.stats[request.get("category", "unknown")]["handled"] += 1
            outcome = {
                "action": "HANDLED",
                "confidence": escalation.confidence,
            }

        self.stats[request.get("category", "unknown")]["total"] += 1
        return outcome

    def run_quality_review(self, budget: int = 50) -> dict:
        """Run stratified sampling on handled decisions."""
        if not self.handled:
            return {"error": "No handled decisions to review"}

        # Stratified sample from handled decisions
        sampled = stratified_sample(self.handled, budget)
        errors_found = sum(1 for d in sampled if d.get("has_error", False))

        return {
            "reviewed": len(sampled),
            "errors_found": errors_found,
            "error_rate": errors_found / len(sampled) if sampled else 0,
            "category_breakdown": {
                cat: sum(1 for d in sampled if d.get("category") == cat)
                for cat in categories
            },
        }

    def summary(self) -> dict:
        """Pipeline performance summary."""
        return {
            "total_processed": len(self.handled) + len(self.escalated),
            "handled_by_agent": len(self.handled),
            "escalated_to_human": len(self.escalated),
            "escalation_rate": (
                len(self.escalated) /
                (len(self.handled) + len(self.escalated))
                if (self.handled or self.escalated) else 0
            ),
            "per_category_stats": dict(self.stats),
        }


# Run the pipeline on a batch of requests
pipeline = EscalationPipeline()

# Create a batch of 500 diverse requests
random.seed(42)
batch_requests = []
for _ in range(500):
    cat = random.choices(
        list(categories.keys()),
        weights=[c["count"] for c in categories.values()],
        k=1
    )[0]
    cat_info = categories[cat]
    is_error = random.random() < cat_info["actual_error_rate"]

    # Generate request characteristics based on category
    request = {
        "message": random.choice([
            "Check my order status",
            "I need help with my claim",
            "What does my policy cover?",
            "I want to speak to a manager",
            "My lawyer will contact you about this",
            "I need to file a large claim",
            "This is the third time I've called",
            "Please update my address",
            "Does my policy cover this situation?",
            "Cancel my subscription",
        ]),
        "claim_value": random.uniform(50, 300000) if cat == "high_value"
                       else random.uniform(10, 5000),
        "prior_attempts": random.randint(0, 4) if cat == "edge_cases"
                         else random.randint(0, 1),
        "policy_match_confidence": random.uniform(0.3, 0.6) if cat == "edge_cases"
                                   else random.uniform(0.7, 0.99),
        "category": cat,
        "has_error": is_error,
        "context": cat_info["description"],
    }
    batch_requests.append(request)

# Process all requests
for req in batch_requests:
    pipeline.process_request(req)

# Print summary
print("PIPELINE EXECUTION SUMMARY:")
print("=" * 60)
summary = pipeline.summary()
print(f"\n  Total processed: {summary['total_processed']}")
print(f"  Handled by agent: {summary['handled_by_agent']}")
print(f"  Escalated to human: {summary['escalated_to_human']}")
print(f"  Escalation rate: {summary['escalation_rate']:.1%}")

# Run quality review
review = pipeline.run_quality_review(budget=50)
print(f"\n  Quality Review ({review['reviewed']} sampled):")
print(f"    Errors found: {review['errors_found']}")
print(f"    Error rate in sample: {review['error_rate']:.1%}")

print(f"\n  Category breakdown of reviews:")
for cat, count in review['category_breakdown'].items():
    if count > 0:
        print(f"    {cat:25s}: {count} reviews")

In [ ]:
# ============================================================
# 8B: Visualize pipeline flow
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Escalation rate by category
ax = axes[0]
cat_labels = ["Simple", "Medium", "High\nValue", "Edge\nCases", "New\nPolicy"]
cat_keys = list(categories.keys())
esc_rates = []
for k in cat_keys:
    s = pipeline.stats[k]
    total = s["total"] if s["total"] > 0 else 1
    esc_rates.append(s["escalated"] / total * 100)

bar_colors = ["#51cf66", "#74c0fc", "#ffd43b", "#ff922b", "#ff6b6b"]
ax.bar(cat_labels, esc_rates, color=bar_colors, edgecolor="white", linewidth=1.5)
ax.set_ylabel("Escalation Rate (%)", fontsize=11)
ax.set_title("Escalation Rate\nby Category", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(esc_rates) * 1.3 if esc_rates else 10)

# Panel 2: Volume handled vs escalated
ax = axes[1]
handled = summary["handled_by_agent"]
escalated = summary["escalated_to_human"]
ax.pie([handled, escalated],
       labels=[f"Handled\n({handled})", f"Escalated\n({escalated})"],
       colors=["#51cf66", "#ff922b"],
       autopct="%1.1f%%", startangle=90,
       textprops={"fontsize": 11, "fontweight": "bold"})
ax.set_title("Request Distribution", fontsize=13, fontweight="bold")

# Panel 3: Review allocation
ax = axes[2]
review_cats = [k for k in cat_keys if review["category_breakdown"].get(k, 0) > 0]
review_counts = [review["category_breakdown"][k] for k in review_cats]
review_labels = [labels_map.get(k, k) for k in review_cats]
review_colors = [colors_map.get(k, "#999") for k in review_cats]

if review_counts:
    ax.pie(review_counts, labels=review_labels, colors=review_colors,
           autopct="%1.0f%%", startangle=90,
           textprops={"fontsize": 10})
ax.set_title("Quality Review\nAllocation", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

print("The pipeline automatically:")
print("  1. Classifies each request against concrete thresholds")
print("  2. Generates structured handoffs for escalated cases")
print("  3. Allocates quality reviews to high-risk categories")
print("  4. Tracks per-category performance for continuous improvement")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 9: Key Takeaways

### What You Learned

| # | Concept | Key Insight |
|---|---|---|
| 1 | **Concrete thresholds** | "Escalate when uncertain" is useless. Use measurable rules: keyword matches, dollar amounts, confidence scores, attempt counts. |
| 2 | **Five escalation categories** | Safety concerns, customer requests, high-value decisions, policy gaps, repeated failures -- each with a specific priority level. |
| 3 | **Structured handoffs** | A good handoff includes: reason, summary, what was tried, what remains, and a suggested action. The human does not start from scratch. |
| 4 | **Stratified sampling** | Sample by risk category, not randomly. High-risk categories get disproportionately more review attention. |
| 5 | **Adaptive rates** | Track per-category accuracy and adjust sample rates over time. Categories that improve get fewer reviews; struggling categories get more. |
| 6 | **Integrated pipeline** | Classify, handle/escalate, sample, review -- all as one auditable system. |

### Decision Framework

When designing escalation criteria, ask yourself:

1. **Is this rule testable?** Can you write a unit test that verifies the rule fires correctly?
2. **Is this rule measurable?** Does it use a concrete threshold (dollar amount, keyword, count)?
3. **Is this rule auditable?** Can you trace exactly why a decision was made?
4. **Is this rule consistent?** Does it produce the same result regardless of who runs it?
5. **Does the handoff preserve context?** Can a human reviewer start working immediately?

### Exam Relevance

On the Claude Certified Architect exam, escalation questions typically appear as:
- "Given this scenario, should the agent escalate or handle?" (apply the threshold rules)
- "What is missing from this handoff summary?" (check for the 5 handoff components)
- "How would you improve this quality review process?" (random -> stratified sampling)

### What's Next

In **Notebook 3: Error Propagation Across Agents**, we will explore what happens when errors occur in multi-agent systems -- how to classify errors as transient vs permanent, build crash recovery manifests, and implement graceful degradation.

---

*Claude Certified Architect Prep -- Pod 5: Context Management & Reliability*

In [ ]:
# ============================================================
# Final: Section checklist
# ============================================================

print("Notebook 2: Escalation & Human-in-the-Loop")
print("=" * 55)
print()
print("Section Checklist:")
sections = [
    "1. Why Does This Matter (motivation)",
    "2. Building Intuition (judgment exercise)",
    "3. Core Implementation (EscalationClassifier)",
    "4. Exercise: Structured Handoff Protocol",
    "5. Exercise: Stratified Sampling System",
    "6. Exercise: Adaptive Sample Rates",
    "7. Visualization (sampling comparison charts)",
    "8. Mini-Project: Complete Escalation Pipeline",
    "9. Key Takeaways + What's Next",
]
for s in sections:
    print(f"  [x] {s}")

print()
print("Key classes/functions defined:")
functions = [
    "EscalationClassifier.classify()       -- 5-rule escalation decision",
    "generate_handoff_solution()            -- structured handoff generator",
    "score_handoff_solution()               -- handoff quality scorer",
    "stratified_sample()                    -- risk-weighted sampling",
    "classify_decision_complexity_solution() -- category classifier",
    "calculate_adaptive_rates_solution()    -- adaptive rate adjuster",
    "EscalationPipeline                     -- integrated end-to-end system",
]
for f in functions:
    print(f"  - {f}")

print()
print("All code is self-contained and runnable without an API key.")
print("Proceed to Notebook 3: Error Propagation Across Agents when ready.")